In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans


# ========= 直接在这里改参数 =========
input_csv = "syn_data/p0.8_mu0.2_l5_1.csv"
#input_csv="synthetic_temporal_network.csv"
output_csv = "test_l5.csv"

k = 4
n_clusters = 5
time_window = 100
step_decay = 0.8
random_state = 42
undirected = True
# ==================================


def build_temporal_adjacency(df, undirected=True):
    adj = defaultdict(list)

    for row in df.itertuples(index=False):
        s = row.source
        d = row.destination
        t = row.timestamp

        adj[s].append((d, t))
        if undirected:
            adj[d].append((s, t))

    for u in adj:
        adj[u].sort(key=lambda x: x[1])

    return adj


def temporal_transition_probs(neighbors, current_time, time_window):
    """
    给定某个节点的所有时序邻边 neighbors = [(v, t_edge), ...]
    只考虑窗口内的边:
        |t_edge - current_time| <= time_window

    对每个邻居 v，统计该窗口内与当前节点的连边数量 count_v，
    然后按 count_v 归一化作为转移概率。

    返回:
        next_probs = [(v, next_time, prob), ...]
    其中 next_time 取该邻居窗口内距离 current_time 最近的事件时间。
    """
    if not neighbors:
        return []

    neighbor_count = defaultdict(int)
    neighbor_best_time = {}
    neighbor_best_dt = {}

    for v, t_edge in neighbors:
        dt = abs(t_edge - current_time)
        if dt <= time_window:
            neighbor_count[v] += 1

            if (v not in neighbor_best_dt) or (dt < neighbor_best_dt[v]):
                neighbor_best_dt[v] = dt
                neighbor_best_time[v] = t_edge

    if not neighbor_count:
        return []

    total_count = sum(neighbor_count.values())
    if total_count <= 0:
        return []

    next_probs = []
    for v, cnt in neighbor_count.items():
        prob = cnt / total_count
        next_time = neighbor_best_time[v]
        next_probs.append((v, next_time, float(prob)))

    return next_probs


def temporal_k_hop_visit_distribution(start_node, start_time, adj, k, time_window, step_decay=1.0):
    """
    返回前 1..k 步累计访问节点分布，而不是只返回第 k 步终点分布。
    """
    state_dist = {(start_node, start_time): 1.0}
    visit_dist = defaultdict(float)

    for step in range(1, k + 1):
        new_state_dist = defaultdict(float)

        for (u, cur_t), p_state in state_dist.items():
            neighbors = adj.get(u, [])
            trans = temporal_transition_probs(
                neighbors=neighbors,
                current_time=cur_t,
                time_window=time_window
            )

            if not trans:
                new_state_dist[(u, cur_t)] += p_state
                continue

            for v, next_t, p_trans in trans:
                new_state_dist[(v, next_t)] += p_state * p_trans

        state_dist = dict(new_state_dist)

        step_weight = step_decay ** (step - 1)
        for (u, _t), p in state_dist.items():
            visit_dist[u] += step_weight * p

    total = sum(visit_dist.values())
    if total > 0:
        for u in visit_dist:
            visit_dist[u] /= total

    return dict(visit_dist)


def build_node_event_features(df, k, time_window, step_decay, undirected=True):
    all_nodes = pd.Index(
        pd.concat([df["source"], df["destination"]], ignore_index=True).unique()
    )
    node2id = {node: i for i, node in enumerate(all_nodes)}

    adj = build_temporal_adjacency(df, undirected=undirected)

    num_events = len(df)
    num_nodes = len(all_nodes)
    X = np.zeros((2 * num_events, num_nodes), dtype=np.float32)
    meta = []

    for row_idx, row in df.iterrows():
        s = row["source"]
        d = row["destination"]
        t = row["timestamp"]

        s_dist = temporal_k_hop_visit_distribution(
            start_node=s,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay
        )
        feature_row = len(meta)
        for node, prob in s_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "source"))

        d_dist = temporal_k_hop_visit_distribution(
            start_node=d,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay
        )
        feature_row = len(meta)
        for node, prob in d_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "destination"))

    return X, meta, all_nodes


def run():
    df = pd.read_csv(input_csv)

    required_cols = {"source", "destination", "timestamp"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Input CSV 缺少必要列: {missing}")

    df = df.copy()
    df["orig_index"] = np.arange(len(df))
    df = df.sort_values(["timestamp", "orig_index"]).reset_index(drop=True)

    X, meta, nodes = build_node_event_features(
        df=df,
        k=k,
        time_window=time_window,
        step_decay=step_decay,
        undirected=undirected
    )

    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = km.fit_predict(X)

    source_commu = np.full(len(df), -1, dtype=int)
    destination_commu = np.full(len(df), -1, dtype=int)

    for label, (row_idx, role) in zip(labels, meta):
        if role == "source":
            source_commu[row_idx] = int(label)
        else:
            destination_commu[row_idx] = int(label)

    df["source_commu"] = source_commu
    df["destination_commu"] = destination_commu

    df = df.sort_values("orig_index").reset_index(drop=True)

    out_df = df[
        ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    ].copy()
    out_df.to_csv(output_csv, index=False)

    print(f"Done. 输出保存到: {output_csv}")
    print(f"事件数: {len(df)}")
    print(f"节点数: {len(nodes)}")
    print(f"node-event数: {len(meta)}")
    print(
        f"k = {k}, time_window = {time_window}, "
        f"step_decay = {step_decay}, n_clusters = {n_clusters}"
    )


if __name__ == "__main__":
    run()

Done. 输出保存到: test_l20.csv
事件数: 4995
节点数: 50
node-event数: 9990
k = 4, time_window = 100, step_decay = 0.8, n_clusters = 5


In [29]:
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans


# ========= 直接在这里改参数 =========
input_csv = "p0.8_mu0.2_l5.csv"
# input_csv = "synthetic_temporal_network.csv"
output_csv = "test_temp.csv"

k = 3
n_clusters = 10
time_window = 300
step_decay = 1.0
time_decay = 0.01   # 新增：时间差指数衰减系数
random_state = 42
undirected = True
# ==================================


def build_temporal_adjacency(df, undirected=True):
    adj = defaultdict(list)

    for row in df.itertuples(index=False):
        s = row.source
        d = row.destination
        t = row.timestamp

        adj[s].append((d, t))
        if undirected:
            adj[d].append((s, t))

    for u in adj:
        adj[u].sort(key=lambda x: x[1])

    return adj


def temporal_transition_probs(neighbors, current_time, time_window, time_decay):
    """
    给定某个节点的所有时序邻边 neighbors = [(v, t_edge), ...]
    只考虑窗口内满足:
        |t_edge - current_time| <= time_window
    的边。

    对每条候选边 (v, t_edge)，赋予指数衰减权重:
        w = exp(-time_decay * |t_edge - current_time|)

    然后对所有候选边的权重做归一化，得到跳转概率。

    返回:
        next_probs = [(v, next_time, prob), ...]
    这里 next_time 就取对应那条边自己的时间 t_edge。
    """
    if not neighbors:
        return []

    candidates = []
    total_weight = 0.0

    for v, t_edge in neighbors:
        dt = abs(t_edge - current_time)
        if dt <= time_window:
            weight = np.exp(-time_decay * dt)
            if weight > 0:
                candidates.append((v, t_edge, weight))
                total_weight += weight

    if total_weight <= 0:
        return []

    next_probs = []
    for v, next_time, weight in candidates:
        prob = weight / total_weight
        next_probs.append((v, next_time, float(prob)))

    return next_probs


def temporal_k_hop_visit_distribution(start_node, start_time, adj, k, time_window, step_decay=1.0, time_decay=0.01):
    """
    返回前 1..k 步累计访问节点分布，而不是只返回第 k 步终点分布。
    """
    state_dist = {(start_node, start_time): 1.0}
    visit_dist = defaultdict(float)

    for step in range(1, k + 1):
        new_state_dist = defaultdict(float)

        for (u, cur_t), p_state in state_dist.items():
            neighbors = adj.get(u, [])
            trans = temporal_transition_probs(
                neighbors=neighbors,
                current_time=cur_t,
                time_window=time_window,
                time_decay=time_decay
            )

            if not trans:
                new_state_dist[(u, cur_t)] += p_state
                continue

            for v, next_t, p_trans in trans:
                new_state_dist[(v, next_t)] += p_state * p_trans

        state_dist = dict(new_state_dist)

        step_weight = step_decay ** (step - 1)
        for (u, _t), p in state_dist.items():
            visit_dist[u] += step_weight * p

    total = sum(visit_dist.values())
    if total > 0:
        for u in visit_dist:
            visit_dist[u] /= total

    return dict(visit_dist)


def build_node_event_features(df, k, time_window, step_decay, time_decay, undirected=True):
    all_nodes = pd.Index(
        pd.concat([df["source"], df["destination"]], ignore_index=True).unique()
    )
    node2id = {node: i for i, node in enumerate(all_nodes)}

    adj = build_temporal_adjacency(df, undirected=undirected)

    num_events = len(df)
    num_nodes = len(all_nodes)
    X = np.zeros((2 * num_events, num_nodes), dtype=np.float32)
    meta = []

    for row_idx, row in df.iterrows():
        s = row["source"]
        d = row["destination"]
        t = row["timestamp"]

        s_dist = temporal_k_hop_visit_distribution(
            start_node=s,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay,
            time_decay=time_decay
        )
        feature_row = len(meta)
        for node, prob in s_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "source"))

        d_dist = temporal_k_hop_visit_distribution(
            start_node=d,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay,
            time_decay=time_decay
        )
        feature_row = len(meta)
        for node, prob in d_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "destination"))

    return X, meta, all_nodes


def run():
    df = pd.read_csv(input_csv)

    required_cols = {"source", "destination", "timestamp"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Input CSV 缺少必要列: {missing}")

    df = df.copy()
    df["orig_index"] = np.arange(len(df))
    df = df.sort_values(["timestamp", "orig_index"]).reset_index(drop=True)

    X, meta, nodes = build_node_event_features(
        df=df,
        k=k,
        time_window=time_window,
        step_decay=step_decay,
        time_decay=time_decay,
        undirected=undirected
    )

    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = km.fit_predict(X)

    source_commu = np.full(len(df), -1, dtype=int)
    destination_commu = np.full(len(df), -1, dtype=int)

    for label, (row_idx, role) in zip(labels, meta):
        if role == "source":
            source_commu[row_idx] = int(label)
        else:
            destination_commu[row_idx] = int(label)

    df["source_commu"] = source_commu
    df["destination_commu"] = destination_commu

    df = df.sort_values("orig_index").reset_index(drop=True)

    out_df = df[
        ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    ].copy()
    out_df.to_csv(output_csv, index=False)

    print(f"Done. 输出保存到: {output_csv}")
    print(f"事件数: {len(df)}")
    print(f"节点数: {len(nodes)}")
    print(f"node-event数: {len(meta)}")
    print(
        f"k = {k}, time_window = {time_window}, "
        f"step_decay = {step_decay}, time_decay = {time_decay}, "
        f"n_clusters = {n_clusters}"
    )


if __name__ == "__main__":
    run()

Done. 输出保存到: test_temp.csv
事件数: 1980
节点数: 50
node-event数: 3960
k = 3, time_window = 300, step_decay = 1.0, time_decay = 0.01, n_clusters = 10


# Kmeans after PCA

In [30]:
from collections import defaultdict
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


# ========= 直接在这里改参数 =========
input_csv = "p0.8_mu0.2_l5.csv"
k = 5
n_clusters = 5
time_window = 300
step_decay = 1.0
random_state = 42
undirected = True

use_pca = True
pca_n_components = 32
pca_whiten = False
# ==================================


def build_temporal_adjacency(df, undirected=True):
    adj = defaultdict(list)

    for row in df.itertuples(index=False):
        s = row.source
        d = row.destination
        t = row.timestamp

        adj[s].append((d, t))
        if undirected:
            adj[d].append((s, t))

    for u in adj:
        adj[u].sort(key=lambda x: x[1])

    return adj


def temporal_transition_probs(neighbors, current_time, time_window):
    if not neighbors:
        return []

    neighbor_count = defaultdict(int)
    neighbor_best_time = {}
    neighbor_best_dt = {}

    for v, t_edge in neighbors:
        dt = abs(t_edge - current_time)
        if dt <= time_window:
            neighbor_count[v] += 1

            if (v not in neighbor_best_dt) or (dt < neighbor_best_dt[v]):
                neighbor_best_dt[v] = dt
                neighbor_best_time[v] = t_edge

    if not neighbor_count:
        return []

    total_count = sum(neighbor_count.values())
    if total_count <= 0:
        return []

    next_probs = []
    for v, cnt in neighbor_count.items():
        prob = cnt / total_count
        next_time = neighbor_best_time[v]
        next_probs.append((v, next_time, float(prob)))

    return next_probs


def temporal_k_hop_visit_distribution(start_node, start_time, adj, k, time_window, step_decay=1.0):
    state_dist = {(start_node, start_time): 1.0}
    visit_dist = defaultdict(float)

    for step in range(1, k + 1):
        new_state_dist = defaultdict(float)

        for (u, cur_t), p_state in state_dist.items():
            neighbors = adj.get(u, [])
            trans = temporal_transition_probs(
                neighbors=neighbors,
                current_time=cur_t,
                time_window=time_window
            )

            if not trans:
                new_state_dist[(u, cur_t)] += p_state
                continue

            for v, next_t, p_trans in trans:
                new_state_dist[(v, next_t)] += p_state * p_trans

        state_dist = dict(new_state_dist)

        step_weight = step_decay ** (step - 1)
        for (u, _t), p in state_dist.items():
            visit_dist[u] += step_weight * p

    total = sum(visit_dist.values())
    if total > 0:
        for u in visit_dist:
            visit_dist[u] /= total

    return dict(visit_dist)


def build_node_event_features(df, k, time_window, step_decay, undirected=True):
    all_nodes = pd.Index(
        pd.concat([df["source"], df["destination"]], ignore_index=True).unique()
    )
    node2id = {node: i for i, node in enumerate(all_nodes)}

    adj = build_temporal_adjacency(df, undirected=undirected)

    num_events = len(df)
    num_nodes = len(all_nodes)
    X = np.zeros((2 * num_events, num_nodes), dtype=np.float32)
    meta = []

    for row_idx, row in df.iterrows():
        s = row["source"]
        d = row["destination"]
        t = row["timestamp"]

        s_dist = temporal_k_hop_visit_distribution(
            start_node=s,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay
        )
        feature_row = len(meta)
        for node, prob in s_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "source"))

        d_dist = temporal_k_hop_visit_distribution(
            start_node=d,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay
        )
        feature_row = len(meta)
        for node, prob in d_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "destination"))

    return X, meta, all_nodes


def apply_pca_if_needed(X, use_pca=True, pca_n_components=32, pca_whiten=False, random_state=42):
    if not use_pca:
        return X, None

    n_samples, n_features = X.shape
    max_valid_components = min(n_samples, n_features)

    if max_valid_components < 1:
        raise ValueError("PCA 无法执行：输入特征矩阵为空。")

    n_components = min(pca_n_components, max_valid_components)

    pca = PCA(
        n_components=n_components,
        whiten=pca_whiten,
        random_state=random_state
    )
    X_pca = pca.fit_transform(X)

    explained = pca.explained_variance_ratio_.sum()
    print(f"PCA 降维: {X.shape} -> {X_pca.shape}")
    print(f"PCA 累计解释方差比: {explained:.6f}")

    return X_pca, pca


def run():
    df = pd.read_csv(input_csv)

    required_cols = {"source", "destination", "timestamp"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Input CSV 缺少必要列: {missing}")

    df = df.copy()
    df["orig_index"] = np.arange(len(df))
    df = df.sort_values(["timestamp", "orig_index"]).reset_index(drop=True)

    X, meta, nodes = build_node_event_features(
        df=df,
        k=k,
        time_window=time_window,
        step_decay=step_decay,
        undirected=undirected
    )

    print(f"原始特征矩阵形状: {X.shape}")

    X_for_cluster, pca_model = apply_pca_if_needed(
        X,
        use_pca=use_pca,
        pca_n_components=pca_n_components,
        pca_whiten=pca_whiten,
        random_state=random_state
    )

    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = km.fit_predict(X_for_cluster)

    source_commu = np.full(len(df), -1, dtype=int)
    destination_commu = np.full(len(df), -1, dtype=int)

    for label, (row_idx, role) in zip(labels, meta):
        if role == "source":
            source_commu[row_idx] = int(label)
        else:
            destination_commu[row_idx] = int(label)

    df["source_commu"] = source_commu
    df["destination_commu"] = destination_commu

    df = df.sort_values("orig_index").reset_index(drop=True)

    out_df = df[
        ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    ].copy()

    input_name = Path(input_csv).stem
    output_dir = Path("results") / input_name
    output_dir.mkdir(parents=True, exist_ok=True)
    output_csv = output_dir / "our_method.csv"

    out_df.to_csv(output_csv, index=False)

    print(f"Done. 输出保存到: {output_csv}")
    print(f"事件数: {len(df)}")
    print(f"节点数: {len(nodes)}")
    print(f"node-event数: {len(meta)}")
    print(
        f"k = {k}, time_window = {time_window}, "
        f"step_decay = {step_decay}, n_clusters = {n_clusters}, "
        f"use_pca = {use_pca}, pca_n_components = {pca_n_components}"
    )


if __name__ == "__main__":
    run()

原始特征矩阵形状: (3960, 50)
PCA 降维: (3960, 50) -> (3960, 32)
PCA 累计解释方差比: 0.879647
Done. 输出保存到: results/p0.8_mu0.2_l5/our_method.csv
事件数: 1980
节点数: 50
node-event数: 3960
k = 5, time_window = 300, step_decay = 1.0, n_clusters = 5, use_pca = True, pca_n_components = 32


In [43]:
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


# ========= 直接在这里改参数 =========
syn_data_dir = "syn_data"

k = 4
n_clusters = 5
time_window = 100
step_decay = 1.0
random_state = 42
undirected = True

use_pca = False
pca_n_components = 32
pca_whiten = False
# ==================================


def build_temporal_adjacency(df, undirected=True):
    adj = defaultdict(list)

    for row in df.itertuples(index=False):
        s = row.source
        d = row.destination
        t = row.timestamp

        adj[s].append((d, t))
        if undirected:
            adj[d].append((s, t))

    for u in adj:
        adj[u].sort(key=lambda x: x[1])

    return adj


def temporal_transition_probs(neighbors, current_time, time_window):
    if not neighbors:
        return []

    neighbor_count = defaultdict(int)
    neighbor_best_time = {}
    neighbor_best_dt = {}

    for v, t_edge in neighbors:
        dt = abs(t_edge - current_time)
        if dt <= time_window:
            neighbor_count[v] += 1

            if (v not in neighbor_best_dt) or (dt < neighbor_best_dt[v]):
                neighbor_best_dt[v] = dt
                neighbor_best_time[v] = t_edge

    if not neighbor_count:
        return []

    total_count = sum(neighbor_count.values())
    if total_count <= 0:
        return []

    next_probs = []
    for v, cnt in neighbor_count.items():
        prob = cnt / total_count
        next_time = neighbor_best_time[v]
        next_probs.append((v, next_time, float(prob)))

    return next_probs


def temporal_k_hop_visit_distribution(start_node, start_time, adj, k, time_window, step_decay=1.0):
    state_dist = {(start_node, start_time): 1.0}
    visit_dist = defaultdict(float)

    for step in range(1, k + 1):
        new_state_dist = defaultdict(float)

        for (u, cur_t), p_state in state_dist.items():
            neighbors = adj.get(u, [])
            trans = temporal_transition_probs(
                neighbors=neighbors,
                current_time=cur_t,
                time_window=time_window
            )

            if not trans:
                new_state_dist[(u, cur_t)] += p_state
                continue

            for v, next_t, p_trans in trans:
                new_state_dist[(v, next_t)] += p_state * p_trans

        state_dist = dict(new_state_dist)

        step_weight = step_decay ** (step - 1)
        for (u, _t), p in state_dist.items():
            visit_dist[u] += step_weight * p

    total = sum(visit_dist.values())
    if total > 0:
        for u in visit_dist:
            visit_dist[u] /= total

    return dict(visit_dist)


def build_node_event_features(df, k, time_window, step_decay, undirected=True):
    all_nodes = pd.Index(
        pd.concat([df["source"], df["destination"]], ignore_index=True).unique()
    )
    node2id = {node: i for i, node in enumerate(all_nodes)}

    adj = build_temporal_adjacency(df, undirected=undirected)

    num_events = len(df)
    num_nodes = len(all_nodes)
    X = np.zeros((2 * num_events, num_nodes), dtype=np.float32)
    meta = []

    for row_idx, row in df.iterrows():
        s = row["source"]
        d = row["destination"]
        t = row["timestamp"]

        s_dist = temporal_k_hop_visit_distribution(
            start_node=s,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay
        )
        feature_row = len(meta)
        for node, prob in s_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "source"))

        d_dist = temporal_k_hop_visit_distribution(
            start_node=d,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay
        )
        feature_row = len(meta)
        for node, prob in d_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "destination"))

    return X, meta, all_nodes


def apply_pca_if_needed(X, use_pca=True, pca_n_components=32, pca_whiten=False, random_state=42):
    if not use_pca:
        return X, None

    n_samples, n_features = X.shape
    max_valid_components = min(n_samples, n_features)

    if max_valid_components < 1:
        raise ValueError("PCA 无法执行：输入特征矩阵为空。")

    n_components = min(pca_n_components, max_valid_components)

    pca = PCA(
        n_components=n_components,
        whiten=pca_whiten,
        random_state=random_state
    )
    X_pca = pca.fit_transform(X)

    explained = pca.explained_variance_ratio_.sum()
    print(f"PCA 降维: {X.shape} -> {X_pca.shape}")
    print(f"PCA 累计解释方差比: {explained:.6f}")

    return X_pca, pca


def run(input_csv):
    df = pd.read_csv(input_csv)

    required_cols = {"source", "destination", "timestamp"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Input CSV 缺少必要列: {missing}")

    df = df.copy()
    df["orig_index"] = np.arange(len(df))
    df = df.sort_values(["timestamp", "orig_index"]).reset_index(drop=True)

    X, meta, nodes = build_node_event_features(
        df=df,
        k=k,
        time_window=time_window,
        step_decay=step_decay,
        undirected=undirected
    )

    print(f"原始特征矩阵形状: {X.shape}")

    X_for_cluster, _ = apply_pca_if_needed(
        X,
        use_pca=use_pca,
        pca_n_components=pca_n_components,
        pca_whiten=pca_whiten,
        random_state=random_state
    )

    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = km.fit_predict(X_for_cluster)

    source_commu = np.full(len(df), -1, dtype=int)
    destination_commu = np.full(len(df), -1, dtype=int)

    for label, (row_idx, role) in zip(labels, meta):
        if role == "source":
            source_commu[row_idx] = int(label)
        else:
            destination_commu[row_idx] = int(label)

    df["source_commu"] = source_commu
    df["destination_commu"] = destination_commu

    df = df.sort_values("orig_index").reset_index(drop=True)

    out_df = df[
        ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    ].copy()

    input_name = Path(input_csv).stem
    output_dir = Path("results") / input_name
    output_dir.mkdir(parents=True, exist_ok=True)
    output_csv = output_dir / "our_method.csv"

    out_df.to_csv(output_csv, index=False)

    print(f"Done. 输出保存到: {output_csv}")



if __name__ == "__main__":
    csv_files = sorted(Path(syn_data_dir).glob("*.csv"))

    print(f"在 {syn_data_dir} 中找到 {len(csv_files)} 个 CSV 文件")

    for csv_path in csv_files:
        print("=" * 80)
        print(f"正在处理: {csv_path}")
        try:
            run(str(csv_path))
        except Exception as e:
            print(f"处理失败: {csv_path}")
            print(f"错误信息: {e}")

    print("=" * 80)
    print("全部处理完成")

在 syn_data 中找到 500 个 CSV 文件
正在处理: syn_data/p0.85_mu0.05_l10_1.csv
原始特征矩阵形状: (4970, 50)
Done. 输出保存到: results/p0.85_mu0.05_l10_1/our_method.csv
正在处理: syn_data/p0.85_mu0.05_l10_2.csv
原始特征矩阵形状: (4970, 50)
Done. 输出保存到: results/p0.85_mu0.05_l10_2/our_method.csv
正在处理: syn_data/p0.85_mu0.05_l10_3.csv
原始特征矩阵形状: (4970, 50)
Done. 输出保存到: results/p0.85_mu0.05_l10_3/our_method.csv
正在处理: syn_data/p0.85_mu0.05_l10_4.csv
原始特征矩阵形状: (4970, 50)
Done. 输出保存到: results/p0.85_mu0.05_l10_4/our_method.csv
正在处理: syn_data/p0.85_mu0.05_l10_5.csv
原始特征矩阵形状: (4970, 50)
Done. 输出保存到: results/p0.85_mu0.05_l10_5/our_method.csv
正在处理: syn_data/p0.85_mu0.05_l15_1.csv
原始特征矩阵形状: (7528, 50)
Done. 输出保存到: results/p0.85_mu0.05_l15_1/our_method.csv
正在处理: syn_data/p0.85_mu0.05_l15_2.csv
原始特征矩阵形状: (7528, 50)
Done. 输出保存到: results/p0.85_mu0.05_l15_2/our_method.csv
正在处理: syn_data/p0.85_mu0.05_l15_3.csv
原始特征矩阵形状: (7528, 50)
Done. 输出保存到: results/p0.85_mu0.05_l15_3/our_method.csv
正在处理: syn_data/p0.85_mu0.05_l15_4.csv
原始特征矩阵形状: (7528, 50)
D

# Local node2vec

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.cluster import KMeans

# 需要安装:
# pip install node2vec
from node2vec import Node2Vec


# ========= 直接在这里改参数 =========
input_csv = "syn_data/p0.8_mu0.2_l5_1.csv"
output_csv = "node2vec_l5.csv"

n_clusters = 5
time_window = 100
random_state = 42
undirected = True

# node2vec 参数
embedding_dim = 64
walk_length = 20
num_walks = 50
window = 10
min_count = 1
batch_words = 128
workers = 4   # 为了稳定复现，建议先设为1
p = 1.0
q = 1.0
# ==================================


def build_time_sorted_events(df):
    """
    构建按时间排序的事件列表，便于窗口过滤。
    """
    events = []
    for row in df.itertuples(index=False):
        events.append((row.source, row.destination, row.timestamp))
    return events


def build_local_weighted_graph(events, center_time, time_window, undirected=True):
    """
    在 [center_time - time_window, center_time + time_window] 内构造静态加权图。
    边权重 = 窗口内该边出现次数。
    """
    edge_count = defaultdict(int)

    for s, d, t in events:
        if abs(t - center_time) <= time_window:
            if undirected:
                u, v = sorted((str(s), str(d)))
                edge_count[(u, v)] += 1
            else:
                edge_count[(str(s), str(d))] += 1

    if undirected:
        G = nx.Graph()
    else:
        G = nx.DiGraph()

    for (u, v), w in edge_count.items():
        G.add_edge(u, v, weight=float(w))

    return G


def node2vec_embedding_for_center_node(
    G,
    center_node,
    embedding_dim=64,
    walk_length=20,
    num_walks=50,
    window=10,
    min_count=1,
    batch_words=128,
    workers=1,
    p=1.0,
    q=1.0,
    random_state=42
):
    """
    在局部图 G 上运行 node2vec，并返回 center_node 的 embedding。
    只返回中心节点的 embedding，其他节点的 embedding 不输出。
    """
    center_node = str(center_node)

    # 如果中心节点根本不在图里，加进去
    if center_node not in G:
        G.add_node(center_node)

    # 图太小/没有边时，node2vec 很不稳定，这里直接返回零向量
    if G.number_of_nodes() <= 1 or G.number_of_edges() == 0:
        return np.zeros(embedding_dim, dtype=np.float32)

    # node2vec 会利用边的 weight 属性
    n2v = Node2Vec(
        G,
        dimensions=embedding_dim,
        walk_length=walk_length,
        num_walks=num_walks,
        workers=workers,
        p=p,
        q=q,
        weight_key="weight",
        seed=random_state
    )

    model = n2v.fit(
        window=window,
        min_count=min_count,
        batch_words=batch_words
    )

    if center_node in model.wv:
        return model.wv[center_node].astype(np.float32)
    else:
        return np.zeros(embedding_dim, dtype=np.float32)


def build_node_event_features(
    df,
    time_window,
    undirected=True,
    embedding_dim=64,
    walk_length=20,
    num_walks=50,
    window=10,
    min_count=1,
    batch_words=128,
    workers=1,
    p=1.0,
    q=1.0,
    random_state=42
):
    """
    对每一个 node-event 构建局部静态加权图，运行 node2vec，
    只提取该 node-event 对应中心节点的 embedding。
    """
    events = build_time_sorted_events(df)

    num_events = len(df)
    X = np.zeros((2 * num_events, embedding_dim), dtype=np.float32)
    meta = []

    for row_idx, row in df.iterrows():
        s = row["source"]
        d = row["destination"]
        t = row["timestamp"]

        # source node-event
        G_s = build_local_weighted_graph(
            events=events,
            center_time=t,
            time_window=time_window,
            undirected=undirected
        )
        s_emb = node2vec_embedding_for_center_node(
            G=G_s,
            center_node=s,
            embedding_dim=embedding_dim,
            walk_length=walk_length,
            num_walks=num_walks,
            window=window,
            min_count=min_count,
            batch_words=batch_words,
            workers=workers,
            p=p,
            q=q,
            random_state=random_state
        )
        feature_row = len(meta)
        X[feature_row] = s_emb
        meta.append((row_idx, "source"))

        # destination node-event
        G_d = build_local_weighted_graph(
            events=events,
            center_time=t,
            time_window=time_window,
            undirected=undirected
        )
        d_emb = node2vec_embedding_for_center_node(
            G=G_d,
            center_node=d,
            embedding_dim=embedding_dim,
            walk_length=walk_length,
            num_walks=num_walks,
            window=window,
            min_count=min_count,
            batch_words=batch_words,
            workers=workers,
            p=p,
            q=q,
            random_state=random_state
        )
        feature_row = len(meta)
        X[feature_row] = d_emb
        meta.append((row_idx, "destination"))

        if (row_idx + 1) % 50 == 0 or (row_idx + 1) == len(df):
            print(f"已处理 {row_idx + 1}/{len(df)} 个事件")

    return X, meta


def run():
    df = pd.read_csv(input_csv)

    required_cols = {"source", "destination", "timestamp"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Input CSV 缺少必要列: {missing}")

    df = df.copy()
    df["orig_index"] = np.arange(len(df))
    df = df.sort_values(["timestamp", "orig_index"]).reset_index(drop=True)

    X, meta = build_node_event_features(
        df=df,
        time_window=time_window,
        undirected=undirected,
        embedding_dim=embedding_dim,
        walk_length=walk_length,
        num_walks=num_walks,
        window=window,
        min_count=min_count,
        batch_words=batch_words,
        workers=workers,
        p=p,
        q=q,
        random_state=random_state
    )

    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = km.fit_predict(X)

    source_commu = np.full(len(df), -1, dtype=int)
    destination_commu = np.full(len(df), -1, dtype=int)

    for label, (row_idx, role) in zip(labels, meta):
        if role == "source":
            source_commu[row_idx] = int(label)
        else:
            destination_commu[row_idx] = int(label)

    df["source_commu"] = source_commu
    df["destination_commu"] = destination_commu

    df = df.sort_values("orig_index").reset_index(drop=True)

    out_df = df[
        ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    ].copy()
    out_df.to_csv(output_csv, index=False)

    print(f"\nDone. 输出保存到: {output_csv}")
    print(f"事件数: {len(df)}")
    print(f"node-event数: {len(meta)}")
    print(f"embedding_dim = {embedding_dim}")
    print(f"time_window = {time_window}")
    print(f"n_clusters = {n_clusters}")


if __name__ == "__main__":
    run()